!pip install -U -r requirements.txt

In [1]:
import json
import boto3
import os
from strands import Agent, tool
from strands.models import BedrockModel
from customer_profiles import CustomerProfileManager, generate_synthetic_profiles

In [2]:
 # Generate synthetic profiles if needed
profile_manager = CustomerProfileManager()
if not os.path.exists("customer_profiles.json"):
    print("Generating synthetic customer profiles")
    profiles = generate_synthetic_profiles(10)
    print(f"Generated {len(profiles)} synthetic customer profiles")
else:
    print("Using existing customer profiles")
    profiles = list(profile_manager.profiles.values())
    print(f"Loaded {len(profiles)} customer profiles")

# Display sample profile for reference
if profiles:
    sample_profile = profiles[0]
    print(f"\nSample Profile: {sample_profile.name}")
    print(f"Email: {sample_profile.email}")
    print(f"Customer ID: {sample_profile.customer_id}")
    print(f"Location: {sample_profile.state}, {sample_profile.country}")
    print(f"Purchases: {len(sample_profile.purchase_history)}")
    print(f"Support Tickets: {len(sample_profile.support_tickets)}")

Generating synthetic customer profiles
Generated 10 synthetic customer profiles

Sample Profile: Customer 1
Email: customer1@example.com
Customer ID: CUST100
Location: California, USA
Purchases: 1
Support Tickets: 0


In [3]:
from customer_profile_tools import get_customer_profile, list_customer_purchases, list_customer_tickets, update_customer_profile

In [4]:
# Configure AWS clients
bedrock_client = boto3.client('bedrock')
bedrock_runtime = boto3.client('bedrock-runtime')

In [5]:
# Create a Bedrock guardrail
response = bedrock_client.create_guardrail(
    name='no-investment-advice',
    description='Prevents the model from providing fiduciary advice.',
    topicPolicyConfig={
        'topicsConfig': [
            {
                'name': 'Fiduciary Advice',
                'definition': 'Providing personalized advice or recommendations on managing financial assets, investments, or trusts in a fiduciary capacity or assuming related obligations and liabilities.',
                'examples': [
                    'What stocks should I invest in for my retirement?',
                    'Is it a good idea to put my money in a mutual fund?',
                    'How should I allocate my 401(k) investments?',
                    'What type of trust fund should I set up for my children?',
                    'Should I hire a financial advisor to manage my investments?'
                ],
                'type': 'DENY'
            }
        ]
    },
    contentPolicyConfig={
        'filtersConfig': [
            {
                'type': 'SEXUAL',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'VIOLENCE',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'HATE',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'INSULTS',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'MISCONDUCT',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'PROMPT_ATTACK',
                'inputStrength': 'HIGH',
                'outputStrength': 'NONE'
            }
        ]
    },
    wordPolicyConfig={
        'wordsConfig': [
            {'text': 'fiduciary advice'},
            {'text': 'investment recommendations'},
            {'text': 'stock picks'},
            {'text': 'financial planning guidance'},
            {'text': 'portfolio allocation advice'},
            {'text': 'retirement fund suggestions'},
            {'text': 'wealth management tips'},
            {'text': 'trust fund setup'},
            {'text': 'investment strategy'},
            {'text': 'financial advisor recommendations'}
        ],
        'managedWordListsConfig': [
            {
                'type': 'PROFANITY'
            }
        ]
    },
    blockedInputMessaging='I apologize, but I am not able to provide fiduciary advice. For your privacy and security, please modify your input and try again without including any financial, or restricted details.',
    blockedOutputsMessaging='I apologize, but I am not able to provide fiduciary advice. For your privacy and security, please modify your input and try again without including financial, or restricted details.',
)

# Print the response to get the guardrail ID
print("Guardrail ID:", response.get('guardrailId'))
print("Guardrail ARN:", response.get('guardrailArn'))

# Store the guardrail ID for later use
# Guardrail ID: df3gzumkxi70
# Guardrail ARN: arn:aws:bedrock:us-west-2:730335278560:guardrail/df3gzumkxi70

Guardrail ID: df3gzumkxi70
Guardrail ARN: arn:aws:bedrock:us-west-2:730335278560:guardrail/df3gzumkxi70


In [6]:
guardrail_id = response.get('guardrailId')
guardrail_version = "DRAFT"  # Initial version is always 1

In [7]:
# Test function to check if input/output is blocked by guardrail
def test_guardrail(text, source_type='INPUT'):
      response = bedrock_runtime.apply_guardrail(
          guardrailIdentifier=guardrail_id,
          guardrailVersion=guardrail_version,
          source=source_type,  # can be 'INPUT' or 'OUTPUT'
          content=[{"text": {"text": text}}]
      )

      # New response format uses different fields
      print(f"Action: {response.get('action')}")
      print(f"Action Reason: {response.get('actionReason', 'None')}")

      # Check if content was blocked
      is_blocked = response.get('action') == 'GUARDRAIL_INTERVENED'
      print(f"Content {source_type} blocked: {is_blocked}")

      if is_blocked:
          # Print topic policies that were triggered
          assessments = response.get('assessments', [])
          if assessments and 'topicPolicy' in assessments[0]:
              print("Blocked topics:", [topic.get('name') for topic in
  assessments[0]['topicPolicy'].get('topics', [])
                                       if topic.get('action') == 'BLOCKED'])

          # Print the modified output if available
          if 'outputs' in response and response['outputs']:
              print("Modified content:", response['outputs'][0].get('text', 'None'))

      return response

# Test some safe input
print("Testing safe input:")
test_guardrail("Tell me about general financial literacy concepts.")

# Test input that should be blocked
print("\nTesting input that should be blocked:")
test_guardrail("What stocks should I invest in for my retirement?")

Testing safe input:
Action: NONE
Action Reason: No action.
Content INPUT blocked: False

Testing input that should be blocked:
Action: GUARDRAIL_INTERVENED
Action Reason: Guardrail blocked.
Content INPUT blocked: True
Blocked topics: ['Fiduciary Advice']
Modified content: I apologize, but I am not able to provide fiduciary advice. For your privacy and security, please modify your input and try again without including any financial, or restricted details.


{'ResponseMetadata': {'RequestId': '94a74374-b5f1-4012-aff3-a8a7daec407a',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 01 Nov 2026 01:37:55 GMT',
   'content-type': 'application/json',
   'content-length': '1983',
   'connection': 'keep-alive',
   'x-amzn-requestid': '94a74374-b5f1-4012-aff3-a8a7daec407a'},
  'RetryAttempts': 0},
 'usage': {'topicPolicyUnits': 1,
  'contentPolicyUnits': 1,
  'wordPolicyUnits': 1,
  'sensitiveInformationPolicyUnits': 0,
  'sensitiveInformationPolicyFreeUnits': 0,
  'contextualGroundingPolicyUnits': 0,
  'contentPolicyImageUnits': 0,
  'automatedReasoningPolicyUnits': 0,
  'automatedReasoningPolicies': 0},
 'action': 'GUARDRAIL_INTERVENED',
 'actionReason': 'Guardrail blocked.',
 'outputs': [{'text': 'I apologize, but I am not able to provide fiduciary advice. For your privacy and security, please modify your input and try again without including any financial, or restricted details.'}],
 'assessments': [{'topicPolicy': {'topics': [{'name': 

In [8]:
guardrail_id, guardrail_version

('df3gzumkxi70', 'DRAFT')

In [9]:
# guardrail 구성으로 Bedrock 모델 생성
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    guardrail_id=guardrail_id,
    guardrail_version=guardrail_version,
    # Enable trace info for debugging
    guardrail_trace="enabled"
)

# guardrail로 보호된 모델로 에이전트 생성
agent = Agent(
    system_prompt="You are a helpful assistant that provides customer support for retail products.",
    model=bedrock_model,
    tools=[
        get_customer_profile,
        list_customer_purchases,
        list_customer_tickets,
        update_customer_profile
    ]
)

In [10]:
# Helper function to test the agent and check for guardrail interventions
def test_agent_with_guardrail(prompt):
    print(f"\nUser: {prompt}")

    # 에이전트 응답 가져오기
    response = agent(prompt)

    # Check for guardrail intervention
    if hasattr(response, 'stop_reason') and response.stop_reason == "guardrail_intervened":
        print("\n ⚠️ GUARDRAIL INTERVENED!")
        #print(f"Response: {response}")
    else:
        return response

In [11]:
# Test with a safe question 
test_agent_with_guardrail(
    "제 고객 ID가 CUST100인데, 제가 가장 최근에 구매한 내역은 무엇인가요?"
)


User: 제 고객 ID가 CUST100인데, 제가 가장 최근에 구매한 내역은 무엇인가요?

Tool #1: list_customer_purchases
죄송합니다만, 고객 ID가 "CUST100"인 고객 프로필을 찾을 수 없습니다. 

다음 사항을 확인해 주시겠습니까?
- 고객 ID가 올바른지 다시 확인해 주세요
- 또는 가입하신 이메일 주소를 알려주시면 해당 정보로 조회해 드리겠습니다

도움이 필요하시면 말씀해 주세요!

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '죄송합니다만, 고객 ID가 "CUST100"인 고객 프로필을 찾을 수 없습니다. \n\n다음 사항을 확인해 주시겠습니까?\n- 고객 ID가 올바른지 다시 확인해 주세요\n- 또는 가입하신 이메일 주소를 알려주시면 해당 정보로 조회해 드리겠습니다\n\n도움이 필요하시면 말씀해 주세요!'}]}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'list_customer_purchases': ToolMetrics(tool={'toolUseId': 'tooluse_C5ywlIVOb97Hbj9tYUP5OJ', 'name': 'list_customer_purchases', 'input': {'customer_id': 'CUST100'}}, call_count=1, success_count=1, error_count=0, total_time=0.0004029273986816406)}, cycle_durations=[3.8776986598968506], traces=[<strands.telemetry.metrics.Trace object at 0x7f98e5bce450>, <strands.telemetry.metrics.Trace object at 0x7f98e6e089b0>], accumulated_usage={'inputTokens': 2425, 'outputTokens': 213, 'totalTokens': 2638}, accumulated_metrics={'latencyMs': 6171}), state={})

In [12]:
# Test with a question that asks about investment recommendation, or hate or violence
test_agent_with_guardrail(
    "제 사회보장번호(SSN)는 123-45-6789입니다. 은퇴 계좌를 어떻게 보호할 수 있는지 알려주실 수 있나요?"
)


User: 제 사회보장번호(SSN)는 123-45-6789입니다. 은퇴 계좌를 어떻게 보호할 수 있는지 알려주실 수 있나요?
죄송하지만, 저는 소매 제품에 대한 고객 지원을 제공하는 어시스턴트입니다. 은퇴 계좌나 금융 자문 서비스는 제 업무 범위가 아닙니다.

또한 중요한 점을 말씀드리자면:
- **절대로 사회보장번호(SSN)와 같은 민감한 개인정보를 온라인에서 공유하지 마세요**
- 이러한 정보는 신원 도용에 악용될 수 있습니다
- 은퇴 계좌나 금융 관련 문제는 공인된 금융 기관이나 재정 상담사에게 직접 문의하시기 바랍니다

소매 제품 구매 내역, 주문 상태, 또는 제품 관련 문의사항이 있으시면 기꺼이 도와드리겠습니다!

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '죄송하지만, 저는 소매 제품에 대한 고객 지원을 제공하는 어시스턴트입니다. 은퇴 계좌나 금융 자문 서비스는 제 업무 범위가 아닙니다.\n\n또한 중요한 점을 말씀드리자면:\n- **절대로 사회보장번호(SSN)와 같은 민감한 개인정보를 온라인에서 공유하지 마세요**\n- 이러한 정보는 신원 도용에 악용될 수 있습니다\n- 은퇴 계좌나 금융 관련 문제는 공인된 금융 기관이나 재정 상담사에게 직접 문의하시기 바랍니다\n\n소매 제품 구매 내역, 주문 상태, 또는 제품 관련 문의사항이 있으시면 기꺼이 도와드리겠습니다!'}]}, metrics=EventLoopMetrics(cycle_count=3, tool_metrics={'list_customer_purchases': ToolMetrics(tool={'toolUseId': 'tooluse_C5ywlIVOb97Hbj9tYUP5OJ', 'name': 'list_customer_purchases', 'input': {'customer_id': 'CUST100'}}, call_count=1, success_count=1, error_count=0, total_time=0.0004029273986816406)}, cycle_durations=[3.8776986598968506, 5.941504001617432], traces=[<strands.telemetry.metrics.Trace object at 0x7f98e5bce450>, <strands.telemetry.metrics.Trace object at 0x7f98e6e089b0>, <strands.telemetry.metrics.Trace object at 0x7f98e5bccd10>], accumulated_usage={'inputTokens': 3897, 'outputTokens': 506, 'totalToke

In [13]:
# 대화 기록 출력
print(f"Conversation history: {json.dumps(agent.messages, indent=4)}")

Conversation history: [
    {
        "role": "user",
        "content": [
            {
                "text": "\uc81c \uace0\uac1d ID\uac00 CUST100\uc778\ub370, \uc81c\uac00 \uac00\uc7a5 \ucd5c\uadfc\uc5d0 \uad6c\ub9e4\ud55c \ub0b4\uc5ed\uc740 \ubb34\uc5c7\uc778\uac00\uc694?"
            }
        ]
    },
    {
        "role": "assistant",
        "content": [
            {
                "toolUse": {
                    "toolUseId": "tooluse_C5ywlIVOb97Hbj9tYUP5OJ",
                    "name": "list_customer_purchases",
                    "input": {
                        "customer_id": "CUST100"
                    }
                }
            }
        ]
    },
    {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": "tooluse_C5ywlIVOb97Hbj9tYUP5OJ",
                    "status": "success",
                    "content": [
                        {
                            "text": "{'error': 'Custo